# 앙상블 검색기(Ensemble Retriever)

- `EnsembleRetriever`는 여러 검색기를 결합하여 더 강력한 검색 결과를 제공하는 LangChain의 기능입니다.
- 이 검색기는 다양한 검색 알고리즘의 장점을 활용하여 단일 알고리즘보다 더 나은 성능을 달성할 수 있습니다.

**주요 특징**
1. 여러 검색기 통합: 다양한 유형의 검색기를 입력으로 받아 결과를 결합합니다.
2. 결과 재순위화: [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) 알고리즘을 사용하여 결과의 순위를 조정합니다.
3. 하이브리드 검색: 주로 `sparse retriever`(예: BM25)와 `dense retriever`(예: 임베딩 유사도)를 결합하여 사용합니다.

**장점**
- Sparse retriever: 키워드 기반 검색에 효과적
- Dense retriever: 의미적 유사성 기반 검색에 효과적

이러한 상호 보완적인 특성으로 인해 `EnsembleRetriever`는 다양한 검색 시나리오에서 향상된 성능을 제공할 수 있습니다.

자세한 내용은 [LangChain 공식 문서](https://python.langchain.com/docs/modules/data_connection/retrievers/ensemble)를 참조하세요.


In [3]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()

True

In [4]:
# LangSmith 추적을 설정합니다. https://smith.langchain.com
# !pip install langchain-teddynote
from langchain_teddynote import logging

# 프로젝트 이름을 입력합니다.
logging.langsmith("CH10-Retriever")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH10-Retriever


## **TF-IDF**

TF-IDF는 **문서 내 특정 단어의 중요도를 측정하는 통계적 가중치**입니다. 정보 검색 분야에서 문서의 관련성을 평가하고 순위를 매기는 데 널리 사용됩니다.

TF-IDF는 두 가지 핵심 개념, **TF(Term Frequency)** 와 **IDF(Inverse Document Frequency)** 의 곱으로 계산됩니다.

---

### 1. TF (Term Frequency) - 단어 빈도

TF는 **특정 문서 내에서 단어가 얼마나 자주 등장하는지**를 나타냅니다.

- **계산:** `(특정 문서에서 단어가 나타난 횟수) / (문서의 전체 단어 수)`
    
- **의미:** TF 값이 높을수록 해당 단어는 그 문서에서 중요한 단어일 가능성이 높습니다.
    

예를 들어, "자동차"라는 단어가 100단어로 구성된 문서 A에서 5번, 200단어 문서 B에서 5번 등장했다면,

- 문서 A의 TF: 5/100=0.05
    
- 문서 B의 TF: 5/200=0.025 문서 A에서 "자동차"의 중요도가 더 높다고 판단합니다.
    

---

### 2. IDF (Inverse Document Frequency) - 역문서 빈도

IDF는 **전체 문서 집합에서 단어가 얼마나 희귀한지**를 나타냅니다.

- **계산:** log((전체문서수)/(단어가포함된문서의수))
    
- **의미:**
    
    - 'a', 'the'와 같이 모든 문서에 흔하게 등장하는 단어는 IDF 값이 낮습니다. (중요하지 않음)
        
    - 특정 주제를 나타내는 '인공지능', '머신러닝'과 같이 몇몇 문서에만 등장하는 단어는 IDF 값이 높습니다. (중요함)
        

### TF-IDF = TF × IDF

**TF-IDF**는 이 두 값을 곱하여 최종 점수를 산출합니다.

- **TF-IDF 값이 높다**는 것은 **"해당 단어가 특정 문서에 자주 등장하면서(TF ↑), 전체 문서 집합에서는 흔하지 않은(IDF ↑)"** 단어라는 것을 의미합니다.
    
- 즉, TF-IDF는 **특정 문서의 주제를 가장 잘 나타내는 핵심 단어**를 찾아내는 데 매우 효과적입니다.
    

### 한계

TF-IDF는 단순한 키워드 빈도를 기반으로 하기 때문에 **단어의 의미적 유사성(Semantic Similarity)**을 파악하지 못합니다. 예를 들어, "자동차"와 "승용차"는 의미가 매우 유사하지만, TF-IDF는 이 둘을 별개의 단어로 취급합니다. 이 때문에 의미 기반 검색(Semantic Search)이 등장했습니다.

- `EnsembleRetriever`를 초기화하여 `BM25Retriever`와 `FAISS` 검색기를 결합합니다. 각 검색기의 가중치를 설정됩니다.


## **BM25Retriever**

**BM25 (Best Matching 25)** 는 키워드 기반 검색 모델로, 텍스트 문서와 사용자의 질의(쿼리) 간의 관련성을 점수화하는 알고리즘입니다. 

LangChain에서 BM25Retriever는 이 알고리즘을 사용하여 문서를 검색하는 리트리버입니다.


### BM25의 작동 원리

BM25는 **TF-IDF** (Term Frequency-Inverse Document Frequency)의 개선된 형태로 볼 수 있습니다. 

TF-IDF가 단순히 단어의 빈도와 중요도를 계산한다면, BM25는 다음 두 가지 요소를 추가적으로 고려하여 더 정교한 순위를 매깁니다.

1. **단어 빈도수 포화**:
    
    - 특정 단어가 문서에 많이 나타날수록 그 문서의 관련성이 높다고 볼 수 있지만, 무한정 높아지는 것은 아닙니다.
        
    - 예를 들어, "자동차"라는 단어가 10번 나온 문서와 100번 나온 문서가 있다면, 100번 나온 문서가 더 관련성이 높겠지만, 그 차이가 10배만큼 크지는 않습니다.
        
    - BM25는 이러한 **빈도수 포화(term frequency saturation)**를 고려하여, 일정 횟수 이상 나타난 단어의 가중치 증가분을 줄입니다.
        
2. **문서 길이 정규화**:
    
    - 문서의 길이가 길수록 특정 단어가 나올 확률이 높습니다. 단순히 단어 빈도만으로 점수를 매기면 긴 문서가 짧은 문서보다 유리해지는 불균형이 발생할 수 있습니다.
        
    - BM25는 문서의 길이를 전체 문서 컬렉션의 **평균 문서 길이와 비교**하여 점수를 조정합니다. 짧은 문서에서 키워드가 자주 나타나면 더 높은 점수를 부여하는 방식입니다.
        

### `BM25Retriever`의 특징

- **키워드 검색**: `BM25Retriever`는 문서의 의미(semantic)를 이해하는 것이 아니라, 쿼리에 포함된 **키워드**를 기반으로 관련성을 판단합니다. 따라서 "Semantic Search"와 같은 기술 용어를 검색할 때는 효과적이지만, "파란 하늘"처럼 의미적으로 유사한 문서를 찾는 데는 약합니다.
    
- **빠른 속도**: 벡터 데이터베이스를 구축할 필요가 없기 때문에 **설정이 간단하고 검색 속도가 매우 빠릅니다.** 특히 대규모 문서 컬렉션에서 키워드 기반의 고속 검색이 필요할 때 유용합니다.
    
- **단독 사용 또는 하이브리드 검색**: `BM25Retriever`는 단독으로 사용될 수도 있지만, **`VectorStoreRetriever`와 함께 사용하여** 키워드 검색과 의미 검색을 결합하는 **하이브리드 검색(Hybrid Search)** 방식에 활용될 때 가장 강력한 성능을 발휘합니다.

In [11]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# 샘플 문서 리스트
doc_list = [
    "I like apples",
    "I like apple company",
    "I like apple's iphone",
    "Apple is my favorite company",
    "I like apple's ipad",
    "I like apple's macbook",
]


# bm25 retriever와 faiss retriever를 초기화합니다.
bm25_retriever = BM25Retriever.from_texts(
    doc_list,
)
bm25_retriever.k = 1  # BM25Retriever의 검색 결과 개수를 1로 설정합니다.

embedding = OpenAIEmbeddings()  # OpenAI 임베딩을 사용합니다.
faiss_vectorstore = FAISS.from_texts(
    doc_list,
    embedding,
)
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 1})

# 앙상블 retriever를 초기화합니다.
# 추가 -------------------------------------------------------------------
# 각 retriever의 가중치를 설정합니다.
# bm25_retriever의 가중치를 0.7, faiss_retriever의 가중치를 0.3으로 설정합니다.
# 가중치의 합은 1이 되도록 합니다.
# -----------------------------------------------------------------------
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.7, 0.3],
)

`ensemble_retriever` 객체의 `get_relevant_documents()` 메서드를 호출하여 관련성 높은 문서를 검색합니다.


In [9]:
# 검색 결과 문서를 가져옵니다.
query = "my favorite fruit is apple"
ensemble_result = ensemble_retriever.invoke(query)
bm25_result = bm25_retriever.invoke(query)
faiss_result = faiss_retriever.invoke(query)

# 가져온 문서를 출력합니다.
print("[Ensemble Retriever]")
for doc in ensemble_result:
    print(f"Content: {doc.page_content}")
    print()

print("[BM25 Retriever]")
for doc in bm25_result:
    print(f"Content: {doc.page_content}")
    print()

print("[FAISS Retriever]")
for doc in faiss_result:
    print(f"Content: {doc.page_content}")
    print()

[Ensemble Retriever]
Content: I like apples

Content: Apple is my favorite company

[BM25 Retriever]
Content: Apple is my favorite company

[FAISS Retriever]
Content: I like apples



In [12]:
# 검색 결과 문서를 가져옵니다.
query = "Apple company makes my favorite iphone"
ensemble_result = ensemble_retriever.invoke(query)
bm25_result = bm25_retriever.invoke(query)
faiss_result = faiss_retriever.invoke(query)

# 가져온 문서를 출력합니다.
print("[Ensemble Retriever]")
for doc in ensemble_result:
    print(f"Content: {doc.page_content}")
    print()

print("[BM25 Retriever]")
for doc in bm25_result:
    print(f"Content: {doc.page_content}")
    print()

print("[FAISS Retriever]")
for doc in faiss_result:
    print(f"Content: {doc.page_content}")
    print()

[Ensemble Retriever]
Content: Apple is my favorite company

Content: I like apple's iphone

[BM25 Retriever]
Content: Apple is my favorite company

[FAISS Retriever]
Content: I like apple's iphone



## 런타임 Config 변경

런타임에서도 retriever 의 속성을 변경할 수 있습니다. 이는 `ConfigurableField` 클래스를 사용하여 가능합니다.

- `weights` 매개변수를 `ConfigurableField` 객체로 정의합니다.
  - 필드의 ID는 "ensemble_weights"로 설정합니다.


In [13]:
from langchain_core.runnables import ConfigurableField


ensemble_retriever = EnsembleRetriever(
    # 리트리버 목록을 설정합니다. 여기서는 bm25_retriever와 faiss_retriever를 사용합니다.
    retrievers=[bm25_retriever, faiss_retriever],
).configurable_fields(
    weights=ConfigurableField(
        # 검색 매개변수의 고유 식별자를 설정합니다.
        id="ensemble_weights",
        # 검색 매개변수의 이름을 설정합니다.
        name="Ensemble Weights",
        # 검색 매개변수에 대한 설명을 작성합니다.
        description="Ensemble Weights",
    )
)

- 검색 시 `config` 매개변수를 통해 검색 설정을 지정합니다.
  - `ensemble_weights` 옵션의 가중치를 [1, 0]으로 설정하여 **모든 검색 결과의 가중치가 BM25 retriever 에 더 많이 부여** 되도록 합니다.

In [14]:
config = {"configurable": {"ensemble_weights": [1, 0]}}

# config 매개변수를 사용하여 검색 설정을 지정합니다.
docs = ensemble_retriever.invoke("my favorite fruit is apple", config=config)
docs  # 검색 결과인 docs를 출력합니다.

[Document(metadata={}, page_content='Apple is my favorite company'),
 Document(id='80a7fe93-f41f-446c-adb4-449d0653fbdb', metadata={}, page_content='I like apples')]

이번에는 검색시 모든 검색 결과의 가중치가 **FAISS retriever 에 더 많이 부여** 되도록 합니다.

In [15]:
config = {"configurable": {"ensemble_weights": [0, 1]}}

# config 매개변수를 사용하여 검색 설정을 지정합니다.
docs = ensemble_retriever.invoke("my favorite fruit is apple", config=config)
docs  # 검색 결과인 docs를 출력합니다.

[Document(id='80a7fe93-f41f-446c-adb4-449d0653fbdb', metadata={}, page_content='I like apples'),
 Document(metadata={}, page_content='Apple is my favorite company')]